
# **K-Nearest Neighbors on the Banknote Authentication Dataset**

In this notebook we implement and explore **K-Nearest Neighbors (KNN)** using our implementation from the `rice_ml` package.

We will:

- Load the **Banknote Authentication** dataset from the UCI repository
- Perform basic **exploratory data analysis (EDA)**
- Preprocess the data with our own utilities
- Train a **KNN** classifier and tune the value of $k$
- Evaluate using classification metrics
- Visualize decision boundaries and the effect of $k$



---

## 1. Environment & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from rice_ml.supervised_learning.knn import KNN
from rice_ml.processing.preprocessing import standardize, train_test_split
from rice_ml.processing.post_processing import accuracy_score, confusion_matrix



---

## 2. Load the Banknote Authentication Dataset

We use the **Banknote Authentication** dataset from the UCI Machine Learning Repository.
Data were extracted from images of genuine and forged banknotes using a **Wavelet Transform** tool.

The dataset has:

- **4 numeric features** derived from wavelet-transformed images
- **1 target**: `class` — `0` = genuine, `1` = forged

### Feature Summary

All features are continuous numerical statistics of the wavelet-transformed image.

| Feature | Description |
|---------|-------------|
| variance | Variance of wavelet transformed image |
| skewness | Skewness of wavelet transformed image |
| curtosis | Curtosis of wavelet transformed image |
| entropy | Entropy of the image |

### Data Notes
- All features are continuous
- No missing values
- Features have different scales, motivating standardization
- 1,372 total samples — well-suited for KNN


In [ ]:

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00267/data_banknote_authentication.txt"
col_names = ['variance', 'skewness', 'curtosis', 'entropy', 'class']
df = pd.read_csv(url, header=None, names=col_names)

print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head(10)



---

## 3. Exploratory Data Analysis

We examine basic statistics, class distribution, feature distributions by class,
and pairwise relationships between features.


In [ ]:

df.describe().round(3)


In [ ]:

print("Missing values per column:")
print(df.isnull().sum())


### Class Distribution

The dataset is reasonably balanced between genuine and forged banknotes.


In [ ]:

label_map = {0: 'Genuine', 1: 'Forged'}
counts = df['class'].map(label_map).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = ['#70A9E0', '#E07070']

axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, f'{v}  ({v/len(df)*100:.1f}%)', ha='center', fontsize=10)
axes[0].set_title('Class Distribution', fontweight='bold')
axes[0].set_ylabel('Count')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Class Proportion', fontweight='bold')

plt.suptitle('Banknote Authentication — Target Variable', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


### Feature Distributions by Class

We compare the distribution of each feature between genuine (0) and forged (1) banknotes.
Strong separation in any feature indicates it will be informative for KNN.


In [ ]:

feature_cols = ['variance', 'skewness', 'curtosis', 'entropy']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, feat in enumerate(feature_cols):
    for lv, ln, c in [(0, 'Genuine', '#70A9E0'), (1, 'Forged', '#E07070')]:
        axes[i].hist(df[feat][df['class'] == lv], bins=30, alpha=0.65,
                     label=ln, color=c, edgecolor='white', density=True)
    axes[i].set_title(feat.title(), fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].legend(fontsize=8)

plt.suptitle('Feature Distributions by Class', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


### Correlation Matrix


In [ ]:

corr = df.corr(numeric_only=True)

plt.figure(figsize=(6, 5))
im = plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.xticks(ticks=np.arange(len(corr.columns)), labels=corr.columns, rotation=45, ha='right')
plt.yticks(ticks=np.arange(len(corr.columns)), labels=corr.columns)
for i in range(len(corr)):
    for j in range(len(corr.columns)):
        plt.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=9)
plt.title('Correlation Matrix (Banknote Features)', fontweight='bold')
plt.tight_layout()
plt.show()


### Scatter Plot — Top Two Features

`variance` and `skewness` are the two most separating features visually.


In [ ]:

plt.figure(figsize=(7, 5))
for lv, ln, c, mk in [(0, 'Genuine', '#70A9E0', 'o'), (1, 'Forged', '#E07070', '^')]:
    mask = df['class'] == lv
    plt.scatter(df['variance'][mask], df['skewness'][mask],
                c=c, marker=mk, alpha=0.5, edgecolors='white',
                linewidths=0.4, s=40, label=ln)
plt.xlabel('Variance')
plt.ylabel('Skewness')
plt.title('Variance vs Skewness by Class', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()



---

## 4. Mathematical Background — K-Nearest Neighbors

**KNN** is a non-parametric, instance-based learning algorithm. It makes predictions by finding the $k$ closest training points to a query point and taking a majority vote.

### Distance Metric

The most common choice is **Euclidean distance**:

$$d(\mathbf{x}, \mathbf{x'}) = \sqrt{\sum_{j=1}^{d} (x_j - x'_j)^2}$$

### Prediction Rule

Given a test point $\mathbf{x}_0$:

1. Find the $k$ training points with smallest distance to $\mathbf{x}_0$ — call this set $\mathcal{N}_k(\mathbf{x}_0)$
2. Predict the **majority class** among the neighbors:

$$\hat{y} = \underset{c}{\arg\max} \sum_{(\mathbf{x}_i, y_i) \in \mathcal{N}_k} \mathbf{1}[y_i = c]$$

### The Role of $k$

| $k$ value | Effect |
|-----------|--------|
| Small $k$ (e.g. 1) | Low bias, high variance — overfits, sensitive to noise |
| Large $k$ | High bias, low variance — smoother boundary, may underfit |
| Optimal $k$ | Balances bias–variance tradeoff — found via cross-validation |

### Why Standardization Matters

KNN is purely distance-based. Features with large scales dominate the distance computation.
Standardizing ensures all features contribute equally:

$$X_{\text{std}} = \frac{X - \mu_{\text{train}}}{\sigma_{\text{train}}}$$



---

## 5. Preprocessing


In [ ]:

X = df[feature_cols].values
y = df['class'].values

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train size : {X_train.shape[0]} samples")
print(f"Test size  : {X_test.shape[0]}  samples")
print(f"Train — Genuine: {np.sum(y_train==0)}, Forged: {np.sum(y_train==1)}")
print(f"Test  — Genuine: {np.sum(y_test==0)},  Forged: {np.sum(y_test==1)}")


In [ ]:

X_train_s, X_test_s = standardize(X_train, X_test)

print("Before standardization — variance feature:")
print(f"  mean = {X_train[:, 0].mean():.4f},  std = {X_train[:, 0].std():.4f}")
print("After standardization:")
print(f"  mean = {X_train_s[:, 0].mean():.4f},  std = {X_train_s[:, 0].std():.4f}")



---

## 6. Train the KNN Model

We start with $k = 5$, a common default choice.


In [ ]:

knn = KNN(k=5)
knn.fit(X_train_s, y_train)

print("KNN model trained with k =" , knn.k)



---

## 7. Evaluate on Training & Test Sets


In [ ]:

y_train_pred = knn.predict(X_train_s)
y_test_pred  = knn.predict(X_test_s)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc  = accuracy_score(y_test,  y_test_pred)

print(f"Training Accuracy : {train_acc:.4f}  ({train_acc*100:.2f}%)")
print(f"Test Accuracy     : {test_acc:.4f}  ({test_acc*100:.2f}%)")


In [ ]:

cm = confusion_matrix(y_test, y_test_pred)
print("Confusion Matrix (rows=True, cols=Predicted):")
print(cm)

tn, fp, fn, tp = cm.ravel()
precision   = tp / (tp + fp)   if (tp + fp)   > 0 else 0
recall      = tp / (tp + fn)   if (tp + fn)   > 0 else 0
f1          = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0
specificity = tn / (tn + fp)   if (tn + fp)   > 0 else 0

print()
print(f"True  Negatives (Genuine → Genuine) : {tn}")
print(f"False Positives (Genuine → Forged)  : {fp}  ← Type I Error")
print(f"False Negatives (Forged  → Genuine) : {fn}  ← Type II Error")
print(f"True  Positives (Forged  → Forged)  : {tp}")
print()
print(f"Precision   : {precision:.4f}")
print(f"Recall      : {recall:.4f}")
print(f"F1 Score    : {f1:.4f}")
print(f"Specificity : {specificity:.4f}")



---

## 8. Tuning $k$ — The Role of the Hyperparameter

We sweep over a range of $k$ values and record train/test accuracy to find the optimal $k$.
This directly demonstrates the **bias–variance tradeoff**:

- Very small $k$ → low train error but high test error (overfitting)
- Very large $k$ → both errors converge (underfitting)


In [ ]:

k_values   = list(range(1, 31))
train_accs = []
test_accs  = []

for k in k_values:
    model = KNN(k=k)
    model.fit(X_train_s, y_train)
    train_accs.append(accuracy_score(y_train, model.predict(X_train_s)))
    test_accs.append(accuracy_score(y_test,  model.predict(X_test_s)))

best_k   = k_values[np.argmax(test_accs)]
best_acc = max(test_accs)
print(f"Best k = {best_k}  →  Test Accuracy = {best_acc:.4f}")


In [ ]:

plt.figure(figsize=(9, 5))
plt.plot(k_values, train_accs, 'o-', color='steelblue', label='Train Accuracy', linewidth=2)
plt.plot(k_values, test_accs,  's-', color='tomato',    label='Test Accuracy',  linewidth=2)
plt.axvline(best_k, color='gray', linestyle='--', linewidth=1.2,
            label=f'Best k = {best_k}')
plt.xlabel('k (Number of Neighbors)')
plt.ylabel('Accuracy')
plt.title('KNN Accuracy vs k — Banknote Authentication', fontweight='bold')
plt.legend()
plt.xticks(k_values)
plt.tight_layout()
plt.show()



---

## 9. Visualizations

### Confusion Matrix Heatmap (best k)


In [ ]:

# Retrain with best k
knn_best = KNN(k=best_k)
knn_best.fit(X_train_s, y_train)
y_best_pred = knn_best.predict(X_test_s)

cm_best = confusion_matrix(y_test, y_best_pred)
class_labels = ['Genuine', 'Forged']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Counts
im0 = axes[0].imshow(cm_best, cmap='Blues')
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(class_labels)
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(class_labels)
axes[0].set_xlabel('Predicted Label', fontsize=11)
axes[0].set_ylabel('True Label', fontsize=11)
axes[0].set_title(f'Confusion Matrix — k={best_k} (Counts)', fontweight='bold')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm_best[i, j]), ha='center', va='center',
                     fontsize=20, fontweight='bold',
                     color='white' if cm_best[i, j] > cm_best.max()/2 else 'black')

# Normalized
cm_norm = cm_best.astype(float) / cm_best.sum(axis=1, keepdims=True)
im1 = axes[1].imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(class_labels)
axes[1].set_yticks([0, 1]); axes[1].set_yticklabels(class_labels)
axes[1].set_xlabel('Predicted Label', fontsize=11)
axes[1].set_ylabel('True Label', fontsize=11)
axes[1].set_title(f'Confusion Matrix — k={best_k} (Normalized)', fontweight='bold')
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f'{cm_norm[i, j]:.2%}', ha='center', va='center',
                     fontsize=16, fontweight='bold',
                     color='white' if cm_norm[i, j] > 0.5 else 'black')

plt.suptitle('Model Evaluation — Banknote Authentication (KNN)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


### 2-D Decision Boundary (Variance vs Skewness)

We plot the decision boundary on the two most discriminative features
for visual intuition about how KNN partitions the feature space.


In [ ]:

# Use only the first two standardized features for visualization
X_train_2d = X_train_s[:, :2]
X_test_2d  = X_test_s[:,  :2]

knn_2d = KNN(k=best_k)
knn_2d.fit(X_train_2d, y_train)

h = 0.05
X_all_2d = np.vstack([X_train_2d, X_test_2d])
xx, yy = np.meshgrid(
    np.arange(X_all_2d[:, 0].min() - 0.5, X_all_2d[:, 0].max() + 0.5, h),
    np.arange(X_all_2d[:, 1].min() - 0.5, X_all_2d[:, 1].max() + 0.5, h)
)
Z = knn_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(9, 6))
plt.contourf(xx, yy, Z, alpha=0.4, cmap=plt.cm.RdBu)
plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=1.5, linestyles='--')

all_X = np.vstack([X_train_2d, X_test_2d])
all_y = np.concatenate([y_train, y_test])
for lv, ln, mk, c in [(0, 'Genuine', 'o', '#2471A3'), (1, 'Forged', '^', '#C0392B')]:
    mask = all_y == lv
    plt.scatter(all_X[mask, 0], all_X[mask, 1],
                c=c, marker=mk, edgecolors='white', s=50,
                linewidths=0.4, alpha=0.8, label=ln)

plt.xlabel('Variance (standardized)', fontsize=12)
plt.ylabel('Skewness (standardized)', fontsize=12)
plt.title(f'KNN Decision Boundary — k={best_k} (2-D projection)', fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


### Decision Boundary Comparison — Effect of $k$

We visualize decision boundaries for different $k$ values side by side to see
how the boundary smooths out as $k$ increases.


In [ ]:

k_compare = [1, 5, 15, 25]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, k in zip(axes, k_compare):
    m = KNN(k=k)
    m.fit(X_train_2d, y_train)
    Z = m.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    acc = accuracy_score(y_test, m.predict(X_test_2d))

    ax.contourf(xx, yy, Z, alpha=0.35, cmap=plt.cm.RdBu)
    for lv, c, mk in [(0, '#2471A3', 'o'), (1, '#C0392B', '^')]:
        mask = all_y == lv
        ax.scatter(all_X[mask, 0], all_X[mask, 1],
                   c=c, marker=mk, edgecolors='white', s=25,
                   linewidths=0.3, alpha=0.7)
    ax.set_title(f'k = {k}\nTest Acc = {acc:.3f}', fontweight='bold', fontsize=11)
    ax.set_xlabel('Variance (std)')
    ax.set_ylabel('Skewness (std)')

plt.suptitle('Decision Boundary for Different Values of k', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()



---

## 10. Model Summary


In [ ]:

tn, fp, fn, tp = cm_best.ravel()
precision   = tp / (tp + fp)   if (tp + fp)   > 0 else 0
recall      = tp / (tp + fn)   if (tp + fn)   > 0 else 0
f1          = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0
specificity = tn / (tn + fp)   if (tn + fp)   > 0 else 0

print("KNN — Banknote Authentication Summary")
print("-" * 40)
print(f"Best k           : {best_k}")
print(f"Test Accuracy    : {accuracy_score(y_test, y_best_pred):.4f}")
print(f"Precision        : {precision:.4f}")
print(f"Recall           : {recall:.4f}")
print(f"F1 Score         : {f1:.4f}")
print(f"Specificity      : {specificity:.4f}")
print("-" * 40)


### Model Limitations

KNN is a simple and highly effective classifier when the dataset is well-separated in feature space.
However, it has important limitations:

- **Computational cost**: prediction requires computing distances to all training points — $O(n \cdot d)$ per query. Slow for large datasets.
- **No explicit model**: KNN stores the entire training set (lazy learner), which is memory-intensive.
- **Sensitive to irrelevant features**: all features contribute equally to distance regardless of relevance.
- **Curse of dimensionality**: in high dimensions, distances become less meaningful and neighbors become less informative.



---

## 11. Conclusion

In this notebook, we:

- Loaded the **Banknote Authentication** dataset from UCI
- Explored class distribution, feature distributions, and feature correlations
- Standardized features and split data using `rice_ml.processing.preprocessing`
- Trained a **custom KNN** classifier from `rice_ml.supervised_learning`
- Swept over $k = 1$ to $30$ to find the best hyperparameter
- Evaluated with `accuracy_score` and `confusion_matrix` from `rice_ml.processing.post_processing`
- Visualized the decision boundary and the effect of $k$ on classification behavior

### Comparison with Pima Diabetes (Logistic Regression) and Boston (Linear Regression)

| Aspect | Boston Housing | Pima Diabetes (LR) | Banknote Auth (KNN) |
|--------|---------------|---------------------|----------------------|
| Algorithm | Linear Regression | Logistic Regression | K-Nearest Neighbors |
| Task | Regression | Classification | Classification |
| Domain | Real estate | Healthcare | Forensic finance |
| Features | 13 | 8 | 4 |
| Samples | 506 | 768 | 1,372 |
| Key metric | R² | Accuracy / AUC | Accuracy / F1 |
